In [2]:
import xarray as xr

In [3]:
ds_om4 = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_5daily_v0.2.1_with_hfds_anomalies"
)
static_data_vars = ["sea_surface_fraction", "hfgeou"]

In [4]:
ds_static_ssf = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/cm4_sea_surface_fraction_fixed.zarr"
)
ds_static_hfgeou = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/cm4_geothermal_hf_fixed.zarr"
)

In [5]:
ds_cm4 = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/cm4_piControl_ocean_200yr_full_chunked.zarr"
)

In [6]:
ds_cm4["sea_surface_fraction"]

<xarray.DataArray 'sea_surface_fraction' (time: 14600, lat: 180, lon: 360)> Size: 4GB
dask.array<open_dataset-sea_surface_fraction, shape=(14600, 180, 360), dtype=float32, chunksize=(10, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 1kB -89.24 -88.25 -87.25 -86.26 ... 87.25 88.25 89.24
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time     (time) object 117kB 0151-01-06 00:00:00 ... 0351-01-01 00:00:00
Attributes:
    cell_measures:  area: areacello
    cell_methods:   time: point
    long_name:      0 if land, 1 if ocean at tracer points

In [7]:
ds_cm4["hfgeou"]

<xarray.DataArray 'hfgeou' (time: 14600, lat: 180, lon: 360)> Size: 4GB
dask.array<open_dataset-hfgeou, shape=(14600, 180, 360), dtype=float32, chunksize=(10, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float64 1kB -89.24 -88.25 -87.25 -86.26 ... 87.25 88.25 89.24
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time     (time) object 117kB 0151-01-06 00:00:00 ... 0351-01-01 00:00:00
Attributes:
    cell_methods:   area:mean yh:mean xh:mean time: point
    long_name:      Upward geothermal heat flux at sea floor
    standard_name:  upward_geothermal_heat_flux_at_sea_floor
    units:          W m-2

In [8]:
static_data = ds_cm4[static_data_vars]

if "time" in static_data.dims:
    static_data = static_data.isel(time=0)

if "lat" in static_data.dims and "lat" not in ds_om4.dims:
    static_data = static_data.rename({"lat": "y", "lon": "x"})

static_data

<xarray.Dataset> Size: 523kB
Dimensions:               (y: 180, x: 360)
Coordinates:
  * y                     (y) float64 1kB -89.24 -88.25 -87.25 ... 88.25 89.24
  * x                     (x) float64 3kB 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
    time                  object 8B 0151-01-06 00:00:00
Data variables:
    sea_surface_fraction  (y, x) float32 259kB dask.array<chunksize=(180, 360), meta=np.ndarray>
    hfgeou                (y, x) float32 259kB dask.array<chunksize=(180, 360), meta=np.ndarray>
Attributes:
    history:        Dataset computed by full-model/scripts/data_process/compu...
    regrid_method:  conservative

In [9]:
ds2_aligned, ds1_aligned = xr.align(ds_om4, static_data, join="exact")
ds_om4_merged = xr.merge([ds1_aligned, ds2_aligned])
ds_om4_merged

<xarray.Dataset> Size: 100GB
Dimensions:               (y: 180, x: 360, time: 4745, lev: 19, y_b: 181,
                           x_b: 361)
Coordinates:
  * y                     (y) float64 1kB -89.24 -88.25 -87.25 ... 88.25 89.24
  * x                     (x) float64 3kB 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
  * time                  (time) object 38kB 1958-01-03 12:00:00 ... 2022-12-...
    areacello             (y, x) float64 518kB dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz                    (lev) int64 152B dask.array<chunksize=(19,), meta=np.ndarray>
    lat                   (y, x) float64 518kB dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b                 (y_b, x_b) float64 523kB dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev                   (lev) float64 152B 2.5 10.0 22.5 ... 4e+03 5e+03 6e+03
    lon                   (y, x) float64 518kB dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b                 (y_b, x_b) float64 523kB dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction        (lev, y, x) float64 10MB dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
    wetmask               (lev, y, x) bool 1MB dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
Dimensions without coordinates: y_b, x_b
Data variables: (12/83)
    sea_surface_fraction  (y, x) float32 259kB dask.array<chunksize=(180, 360), meta=np.ndarray>
    hfgeou                (y, x) float32 259kB dask.array<chunksize=(180, 360), meta=np.ndarray>
    hfds                  (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    hfds_anomalies        (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1050_0         (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_105_0          (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...                    ...
    vo_lev_5000_0         (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_550_0          (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0         (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_65_0           (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_775_0          (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos                   (time, y, x) float32 1GB dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    history:        Dataset computed by full-model/scripts/data_process/compu...
    regrid_method:  conservative

In [10]:
assert (
    ds_static_ssf.sea_surface_fraction.fillna(0).values
    == ds_om4_merged["sea_surface_fraction"].fillna(0).values
).all()
assert (
    ds_static_hfgeou.hfgeou.fillna(0).values == ds_om4_merged["hfgeou"].fillna(0).values
).all()

In [14]:
ds_om4_merged.to_zarr(
    "/pscratch/sd/s/suryad/data/2025-04-11-3D_data_OM4_5daily_v0.2.1_with_hfds_anomalies_with_static_data.zarr",
    mode="w",
    encoding={var: {"compressor": None} for var in ds_om4_merged.data_vars},
)